In [1]:
import pandas as pd
import openpyxl 
import numpy as np

from pymatgen.core import Composition

In [40]:
df = pd.read_excel("singles_composition.xlsx")
print(df.columns)

Index(['Single Composition', 'Single Eg (eV)'], dtype='str')


In [52]:
def normalize_formula(formula):
    comp = Composition(formula).fractional_composition
    return comp.fractional_composition.formula

# Apply it to the composition column
df["Normalized composition"] = df["Single Composition"].apply(normalize_formula)

# Save the result
df.to_excel("Normalized_Single_Compositions.xlsx", index=False)

print(df.head())

KeyError: 'Single Composition'

In [42]:
composition_column = "Single Composition"

# Check for commas
mask = df[composition_column].astype(str).str.contains(",", regex=False)

if mask.any():
    print(f"Found {mask.sum()} composition(s) containing a comma:\n")
    print(df.loc[mask, [composition_column]])
else:
    print("No commas found in the composition column.")

No commas found in the composition column.


In [43]:
#------------------
# Load dataset 
#------------------
df = pd.read_excel("Normalized_Single_Compositions.xlsx")

compositions_column = "Normalized composition" 
target_column = "Single Eg (eV)"

#------------------
# Determine all elements
#------------------
all_elements = set() 
for formula in df[composition_column]:
    comp = Composition(formula)

    for element in comp.as_dict():
        all_elements.add(element)
all_elements = sorted(all_elements)

#-----------------
# Create fingerprint in dataframe 
#-----------------
fingerprint = pd.DataFrame(
    0.0,
    index=df.index,
    columns=all_elements
)

#---------------
# Fill fingerprint
#---------------
for i, formula in enumerate(df[composition_column]):
    comp = Composition(formula)

    for element, amount in comp.as_dict().items():
        fingerprint.loc[i, element] = amount

#---------------
# Add target
#--------------

fingerprint["BandGap"] = df[target_column]

#--------------
# Save
#--------------
fingerprint.to_excel("Chemical_Fingerprint_Singles.xlsx", index=False)

print(fingerprint.head())

     Ag   Al   As   Au    B   Ba   Be   Bi   Br    C  ...   Tl   Tm    U    V  \
0  0.10  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
1  0.25  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
2  0.20  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
3  0.40  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
4  0.40  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   

     W    Y   Yb   Zn   Zr  BandGap  
0  0.0  0.0  0.0  0.0  0.0     0.29  
1  0.0  0.0  0.0  0.0  0.0     0.39  
2  0.0  0.0  0.0  0.0  0.0     0.54  
3  0.0  0.0  0.0  0.0  0.0     0.42  
4  0.0  0.0  0.0  0.0  0.0     0.19  

[5 rows x 80 columns]


In [44]:
# Load the fingerprint dataset
df = pd.read_excel("Chemical_Fingerprint_Singles.xlsx")

# Choose your threshold
threshold = 0.5      # <-- replace with the value you decided to use

# Create the class column
df["Class"] = np.where(df["BandGap"] >= threshold, -1, 1)

# Save
#df.to_excel("Chemical_Fingerprint_Singles_Class.xlsx", index=False)

# Check the distribution
print(df["Class"].value_counts())

Class
 1    2692
-1    1798
Name: count, dtype: int64


In [46]:
X = df.drop(columns=["BandGap", "Class"])

row_sums = X.sum(axis=1)

print(row_sums.describe())
print(row_sums.head())

count    4490.000000
mean       10.569617
std        12.097832
min         0.101000
25%         4.000000
50%         7.000000
75%        13.000000
max       195.000000
dtype: float64
0    7.00
1    7.00
2    7.05
3    7.00
4    7.10
dtype: float64


In [47]:
print(X.duplicated().sum())

579


In [48]:
constant_columns = X.columns[X.nunique() <= 1]

print(constant_columns)

Index([], dtype='str')


In [49]:
print(df.columns)
print(df.iloc[0])

Index(['Ag', 'Al', 'As', 'Au', 'B', 'Ba', 'Be', 'Bi', 'Br', 'C', 'Ca', 'Cd',
       'Ce', 'Cl', 'Co', 'Cr', 'Cs', 'Cu', 'Dy', 'Er', 'Eu', 'F', 'Fe', 'Ga',
       'Gd', 'Ge', 'H', 'Hf', 'Hg', 'Ho', 'I', 'In', 'Ir', 'K', 'La', 'Li',
       'Lu', 'Mg', 'Mn', 'Mo', 'N', 'Na', 'Nb', 'Nd', 'Ni', 'O', 'Os', 'P',
       'Pb', 'Pd', 'Pr', 'Pt', 'Rb', 'Re', 'Rh', 'Ru', 'S', 'Sb', 'Sc', 'Se',
       'Si', 'Sm', 'Sn', 'Sr', 'Ta', 'Tb', 'Tc', 'Te', 'Th', 'Ti', 'Tl', 'Tm',
       'U', 'V', 'W', 'Y', 'Yb', 'Zn', 'Zr', 'BandGap', 'Class'],
      dtype='str')
Ag         0.10
Al         0.00
As         0.00
Au         0.00
B          0.00
           ... 
Yb         0.00
Zn         0.00
Zr         0.00
BandGap    0.29
Class      1.00
Name: 0, Length: 81, dtype: float64


In [50]:
# Fingerprint only
X = df.drop(columns=["BandGap", "Class"])

# Print only the non-zero entries of the first compound
print(X.iloc[0][X.iloc[0] != 0])

Ag    0.1
Cd    0.8
In    2.1
Te    4.0
Name: 0, dtype: float64


In [51]:
X = df.drop(columns=["BandGap", "Class"])

row_sums = X.sum(axis=1)

largest = row_sums.nlargest(10)

print(largest)

idx = largest.index[0]

print("Row sum:", row_sums[idx])
print(X.loc[idx][X.loc[idx] != 0])

3016    195.0
878     194.0
3638    187.0
3094    151.0
956     150.0
3637    127.0
4168    100.0
2030     99.0
2596     98.0
458      97.0
dtype: float64
Row sum: 195.0
Cl     6.0
Cs    39.0
Ga    53.0
Se    97.0
Name: 3016, dtype: float64


In [8]:
# normalizing to 1 correctly 
input_file = "Normalized_Compositions.xlsx"
df = pd.read_excel(input_file)

# numeric colums only 
numeric_cols = df.select_dtypes(include="number").columns.tolist()

# these NOT normalized
exclude = ["Eg", "Class"]
# add also index column, NOT normalized
element_cols = [c for c in numeric_cols if c not in exclude]

# normalize each row 
row_sums = df[element_cols].sum(axis=1)

# avoid division by zero 
df[element_cols] = df[element_cols].div(row_sums.replace(0, 1), axis=0)

# check the result
print("First 10 rows sums after normalization:")
print(df[element_cols].sum(axis=1).head(10))

print("\nMinimum row sum:", df[element_cols].sum(axis=1).min())
print("Maximum row sum:", df[element_cols].sum(axis=1).max())


#df.to_excel("Normalized1_Compositions_.xlsx", index=False)

First 10 rows sums after normalization:
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
5    1.0
6    1.0
7    1.0
8    1.0
9    0.0
dtype: float64

Minimum row sum: 0.0
Maximum row sum: 1.0


In [5]:
print(df.dtypes)

Single Composition            str
Single Eg (eV)            float64
Normalized composition        str
dtype: object


In [6]:
for col in df.columns:
    print(col, df[col].dtype)

Single Composition str
Single Eg (eV) float64
Normalized composition str


In [12]:
# Read the fingerprint
df = pd.read_excel("Chemical_Fingerprint_Singles.xlsx")

# Keep the BandGap unchanged
element_cols = df.columns.drop("BandGap")

# Calculate the total stoichiometry of each row
row_sums = df[element_cols].sum(axis=1)

# Divide every stoichiometric coefficient by the total
df[element_cols] = df[element_cols].div(row_sums, axis=0)

# Verify
print(df[element_cols].sum(axis=1).head())

# Save
df.to_excel("Chemical_Fingerprint_Singles_Normalized.xlsx", index=False)

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
dtype: float64


In [11]:
print(df.head())

         Ag   Al   As   Au    B   Ba   Be   Bi   Br    C  ...   Tl   Tm    U  \
0  0.014286  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   
1  0.035714  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   
2  0.028369  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   
3  0.057143  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   
4  0.056338  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   

     V    W    Y   Yb   Zn   Zr  BandGap  
0  0.0  0.0  0.0  0.0  0.0  0.0     0.29  
1  0.0  0.0  0.0  0.0  0.0  0.0     0.39  
2  0.0  0.0  0.0  0.0  0.0  0.0     0.54  
3  0.0  0.0  0.0  0.0  0.0  0.0     0.42  
4  0.0  0.0  0.0  0.0  0.0  0.0     0.19  

[5 rows x 80 columns]


In [14]:
# Bandgap < treshold: class 1 (metal)
# Bandgap >= treshold: class -1 (non metal)

# Read the normalized fingerprint
df = pd.read_excel("Chemical_Fingerprint_Singles_Normalized.xlsx")

# Choose a threshold
threshold = 0.5

# Create class labels
df["Class"] = df["BandGap"].apply(lambda x: 1 if x < threshold else -1)

# Check the class distribution
print(df["Class"].value_counts())

# Save
df.to_excel("Chemical_Fingerprint_Singles_Normalized_Classes.xlsx", index=False)


Class
 1    2692
-1    1798
Name: count, dtype: int64
